# 12a — Download all cutouts for Gaia-stable diaObjects

## Purpose

Download and cache on disk the three cutout stamps (Science, Template, Difference)
for every diaSource belonging to the **Gaia-stable** diaObjects identified in
`09b_psfFlux_scienceFlux_templateFlux_GaiaDR3matching.ipynb`.

The list of target diaObjectIds is read directly from the statistics CSV produced
by notebook 09b, filtered to `gaia_status == "Gaia stable"`.
Download is delegated to `fink_download_full_cutouts.py` (imported as a module).

## Inputs

| File | Produced by |
|------|-------------|
| `figs_FINK_BLOCK_LC_09b/median_flux_stats_gaia_all_gaia_groups.csv` | notebook 09b |
| `fink_download_full_cutouts.py` | this directory |

## Output

One directory per diaObject: `fullcutouts_{diaObjectId}/`  
containing `manifest.csv`, `manifest.parquet`, and `cutouts/*.npy`.

---
- **Author:** Sylvie Dagoret-Campagne — IJCLab / IN2P3 / CNRS — Université Paris-Saclay
- **Creation date:** 2026-05-13
- **Subject:** Fink/LSST DIA — Dipole hypothesis — cutout download


## 1. Imports & configuration

In [ ]:
import os
import sys
import warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
print(f"pandas {pd.__version__}  |  numpy {np.__version__}")

In [ ]:
# ── Import fink_download_full_cutouts from the same directory ─────────────────
# The script lives alongside this notebook; add the notebook directory to sys.path
# so it can be imported as a regular module.
NB_DIR = os.path.abspath(".")
if NB_DIR not in sys.path:
    sys.path.insert(0, NB_DIR)

from fink_download_full_cutouts import download_full_cutouts
from pathlib import Path

print("fink_download_full_cutouts imported successfully.")
print(f"download_full_cutouts : {download_full_cutouts}")

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────
FILE_STATS = os.path.join(
    "figs_FINK_BLOCK_LC_09b",
    "median_flux_stats_gaia_all_gaia_groups.csv",
)

# ── Selection criteria ────────────────────────────────────────────────────────
# Only download cutouts for Gaia-stable objects (not the variable control group)
STABLE_STATUS = "Gaia stable"

# Set to True to re-download even if fullcutouts_{id}/ already exists on disk
FORCE_REDOWNLOAD = False

print(f"Stats file : {os.path.abspath(FILE_STATS)}")
print(f"Filtering on gaia_status == '{STABLE_STATUS}'")

## 2. Read the target diaObjectId list from notebook 09b

In [ ]:
if not os.path.exists(FILE_STATS):
    raise FileNotFoundError(
        f"{FILE_STATS} not found.\n"
        "Run notebook 09b_psfFlux_scienceFlux_templateFlux_GaiaDR3matching.ipynb first."
    )

df_stats = pd.read_csv(FILE_STATS)
print(f"Total entries in CSV : {len(df_stats)}")
print(f"gaia_status counts:\n{df_stats['gaia_status'].value_counts().to_string()}")
print()

# Keep only Gaia-stable objects
df_stable = df_stats[df_stats["gaia_status"] == STABLE_STATUS].copy()
df_stable = df_stable.sort_values("rank").reset_index(drop=True)

print(f"Gaia-stable objects selected : {len(df_stable)}")
print()
print(
    df_stable[["rank", "diaObjectId", "n_alerts", "gaia_origin", "gaia_G_mag", "gaia_status"]].to_string(
        index=False
    )
)

## 3. Download cutouts for all Gaia-stable diaObjects

For each object, `download_full_cutouts()` fetches the full list of diaSources
via the Fink API and saves the three `.npy` stamp arrays plus a `manifest.csv`
under `fullcutouts_{diaObjectId}/`.  
Already-downloaded files are skipped automatically (idempotent).

In [ ]:
results = []  # summary of what was done

for _, row in df_stable.iterrows():
    obj_id = int(row["diaObjectId"])
    n_alerts = int(row["n_alerts"])
    origin = row["gaia_origin"]
    g_mag = row["gaia_G_mag"]
    outdir = Path(f"fullcutouts_{obj_id}")

    # Check whether the manifest already exists (download already done)
    manifest_path = outdir / "manifest.csv"
    already_done = manifest_path.exists() and not FORCE_REDOWNLOAD

    print(f"\n{'=' * 60}")
    print(f"diaObjectId = {obj_id}")
    print(f"  gaia_origin : {origin}")
    print(f"  Gaia G      : {g_mag:.3f}")
    print(f"  n_alerts    : {n_alerts}")

    if already_done:
        # Count files already on disk
        n_npy = len(list((outdir / "cutouts").glob("*.npy"))) if (outdir / "cutouts").exists() else 0
        print(f"  → already on disk ({n_npy} .npy files) — skipping.")
        status = "skipped"
    else:
        try:
            download_full_cutouts(
                dia_object_id=obj_id,
                outdir=outdir,
                skip_existing=True,
            )
            status = "downloaded"
        except Exception as exc:
            print(f"  ERROR: {exc}")
            status = f"error: {exc}"

    results.append(
        {
            "diaObjectId": obj_id,
            "gaia_origin": origin,
            "gaia_G_mag": g_mag,
            "n_alerts": n_alerts,
            "outdir": str(outdir),
            "status": status,
        }
    )

print(f"\n{'=' * 60}")
print("All downloads complete.")

## 4. Download summary

In [ ]:
from IPython.display import display

df_results = pd.DataFrame(results)
print("Download summary:")
display(df_results)

print("\nStatus counts:")
print(df_results["status"].value_counts().to_string())

# Disk footprint per object
import glob as _glob

print("\nDisk footprint:")
for _, row in df_results.iterrows():
    npy_files = _glob.glob(os.path.join(row["outdir"], "cutouts", "*.npy"))
    total_mb = sum(os.path.getsize(f) for f in npy_files) / 1e6 if npy_files else 0
    print(f"  {row['diaObjectId']}  {len(npy_files):4d} files  {total_mb:6.1f} MB")